In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

# INITIALIZING SPARK SESSION #

spark = SparkSession.builder.appName("FAERS_PHARMA_ETL_PIPELINE").getOrCreate()

#---------------------------------------------------------------------------------------------------#
 
# PHASE 4 — SILVER LAYER BUSINESS TRANSFORMATION #

# DEMO FILES #

demo_bronze_final_path = "/Volumes/workspace/bronze/faers_parquet/cleaned/demo/"
demo_bronze_df = spark.read.parquet(demo_bronze_final_path)

# AGE STANDARDIZATION #

age_years = (
    F.when(
        F.upper(F.trim(F.col("age_cod"))) == "YR",
        F.col("age").cast("double")
    ).when(
        F.upper(F.trim(F.col("age_cod"))) == "MON",
        F.col("age").cast("double") / 12
    ).when(
        F.upper(F.trim(F.col("age_cod"))) == "WK",
        F.col("age").cast("double") / 52
    ).when(
        F.upper(F.trim(F.col("age_cod"))) == "DY",
        F.col("age").cast("double") / 365
    ).when(
        F.upper(F.trim(F.col("age_cod"))) == "HR",
        F.col("age").cast("double") / 8760
    ).when(
        F.upper(F.trim(F.col("age_cod"))) == "DEC",
        F.col("age").cast("double") * 10
    )
)

age_group = (
    F.when(age_years.isNull(), F.lit(None).cast("string"))
     .when(age_years < 1 / 12, "N")
     .when(age_years < 2, "I")
     .when(age_years < 13, "C")
     .when(age_years < 18, "T")
     .when(age_years < 65, "A")
     .otherwise("E")
)

# BASE STANDARDIZATION #

demo_silver_df = demo_bronze_df.select(
    F.col("primaryid").alias("primary_id"),

    F.when(
        F.col("caseid").isNull(),
        F.col("primaryid")
    ).otherwise(
        F.col("caseid")
    ).alias("case_id"),

    F.col("caseversion").alias("case_version"),
    F.col("i_f_code"),
    F.col("event_dt").alias("event_occurrence_date"),
    F.col("init_fda_dt").alias("init_fda_date"),
    F.col("rept_cod").alias("report_type"),
    F.col("auth_num").alias("nda_num"),
    F.col("mfr_num").alias("manufacture_num"),

    F.round(age_years, 2).alias("patient_age"),
    F.lit("Year").alias("age_unit"),
    age_group.alias("age_grp"),

    F.col("sex"),
    F.col("e_sub").alias("submission_flag"),

    F.round(
    F.when(F.col("wt_cod") == "KG", F.col("wt"))
     .when(F.col("wt_cod") == "LBS", F.col("wt") / 2.20462)
     .otherwise(F.lit(None)),
    2
).alias("weight"),

    F.when(F.col("wt_cod").isin("KG", "LBS"), F.lit("KG"))
 .otherwise(F.lit(None))
 .alias("weight_unit"),
    F.col("rept_dt").alias("report_date"),
    F.col("occp_cod").alias("occupation_code"),
    F.col("reporter_country").alias("reporter_country"),
    F.col("occr_country").alias("occurrence_country")
)

# DRUG FILES#

drug_bronze_final_path = "/Volumes/workspace/bronze/faers_parquet/cleaned/drug/"
drug_bronze_df = spark.read.parquet(drug_bronze_final_path)

# BASE STANDARDIZATION #

drug_silver_df = drug_bronze_df.select(
    F.col("primaryid").alias("primary_id"),
    F.col("caseid").alias("case_id"),
    F.col("drug_seq").alias("drug_sequence"),
    F.col("role_cod").alias("role_of_drug"),
    F.col("drugname").alias("drug_name"),
    F.when(F.col("prod_ai").isNull(),"UNKNOWN").otherwise(F.col("prod_ai")).alias("active_ingredient"),
    F.upper(F.trim(F.col("route"))).alias("route"),
    F.upper(F.trim(F.col("dose_vbm"))).alias("dosage"),
    F.col("cum_dose_chr").alias("cumulative_dosage"),
    F.col("cum_dose_unit").alias("unit"),
    F.when(F.col("dechal").isNull(), "UNKNOWN").otherwise(F.upper(F.trim(F.col("dechal")))).alias("dechallenge_response"),
    F.when(F.col("rechal").isNull(), "UNKNOWN").otherwise(F.upper(F.trim(F.col("rechal")))).alias("rechallenge_response"),
    F.when(F.col("lot_num").isNull(), "NOT_REPORTED").otherwise(F.upper(F.trim(F.col("lot_num")))).alias("lot_number"),
    F.col("exp_dt").alias("expiration_date"),
    F.when(F.col("exp_dt").isNull(), "NOT_REPORTED")
    .otherwise("REPORTED")
    .alias("expiration_date_status"),
    F.coalesce(
    F.col("nda_num").cast("string"),
    F.lit("UNKNOWN")
    ).alias("nda_number"),
    F.col("dose_amt").alias("dose_amount"),
    F.upper(F.trim(F.col("dose_unit"))).alias("dose_unit"),
    F.upper(F.trim(F.col("dose_form"))).alias("dose_form"),
    F.upper(F.trim(F.col("dose_freq"))).alias("dose_frequency")
)

# INDICATION FILES #

indi_bronze_final_path = "/Volumes/workspace/bronze/faers_parquet/cleaned/indi/"
indication_bronze_df = spark.read.parquet(indi_bronze_final_path)

# BASE STANDARDIZATION #

indication_silver_df = indication_bronze_df.select(
    F.col("primaryid").alias("primary_id"),
    F.col("caseid").alias("case_id"),
    F.col("indi_drug_seq").alias("indication_drug_sequence_number"),
    F.col("indi_pt").alias("indication_reason")
)

# REACTION FILES #

reaction_bronze_final_path = "/Volumes/workspace/bronze/faers_parquet/cleaned/reac/"
reaction_bronze_df = spark.read.parquet(reaction_bronze_final_path)

# BASE STANDARDIZATION #

reaction_silver_df = reaction_bronze_df.select(
    F.col("primaryid").alias("primary_id"),
    F.col("caseid").alias("case_id"),
    F.col("pt").alias("reaction"),
    F.col("drug_rec_act").alias("action")
)

# OUTCOME FILES#

outcome_bronze_final_path = "/Volumes/workspace/bronze/faers_parquet/cleaned/outc/"
outcome_bronze_df = spark.read.parquet(outcome_bronze_final_path)

# BASE STANDARDIZATION #

outcome_silver_df = outcome_bronze_df.select(
    F.col("primaryid").alias("primary_id"),
    F.col("caseid").alias("case_id"),
    F.col("outc_cod").alias("outcome_code")
)

# OUTCOME CATEGORIZATION #

outcome_silver_df = outcome_silver_df.withColumn(

    "outcome",

    F.when(F.col("outcome_code") == "DE", "DEATH")
     .when(F.col("outcome_code") == "LT", "LIFE THREATENING")
     .when(F.col("outcome_code") == "HO", "HOSPITALIZATION")
     .when(F.col("outcome_code") == "DS", "DISABILITY")
     .when(F.col("outcome_code") == "CA", "CONGENITAL ANOMALY")
     .when(F.col("outcome_code") == "RI", "REQUIRED INTERVENTION")
     .when(F.col("outcome_code") == "OT", "OTHER SERIOUS")
     .otherwise("OTHERS")
)

# THERAPY FILES #

therapy_bronze_final_path = "/Volumes/workspace/bronze/faers_parquet/cleaned/ther/"
therapy_bronze_df = spark.read.parquet(therapy_bronze_final_path)

# BASE STANDARDIZATION #

therapy_silver_df = therapy_bronze_df.select(
    F.col("primaryid").alias("primary_id"),
    F.col("caseid").alias("case_id"),
    F.col("dsg_drug_seq").alias("drug_sequence_number"),
    F.col("start_dt").alias("start_date"),
    F.col("end_dt").alias("end_date"),
    F.col("dur").alias("duration"),

    # DURATION STANDARDIZATION #
    F.round(
        F.when(F.col("dur_cod") == "YR",  F.col("dur") * 365)
         .when(F.col("dur_cod") == "MON", F.col("dur") * 30)
         .when(F.col("dur_cod") == "WK",  F.col("dur") * 7)
         .when(F.col("dur_cod") == "DY",  F.col("dur"))
         .when(F.col("dur_cod") == "HR",  F.col("dur") / 24)
         .when(F.col("dur_cod") == "MIN", F.col("dur") / 1440)
         .when(F.col("dur_cod") == "DEC", F.col("dur") * 3650)
         .otherwise(None),
        2
    ).alias("therapy_duration_days")
)

# SAVE SILVER LAYER TABLES #

silver_faers_dataframes = {
    "demo": demo_silver_df,
    "drug": drug_silver_df,
    "indication": indication_silver_df,
    "reaction": reaction_silver_df,
    "outcome": outcome_silver_df,
    "therapy": therapy_silver_df,
}

for name, dataframe in silver_faers_dataframes.items():
    dataframe.write.mode("overwrite").parquet(
        f"/Volumes/workspace/silver/faers_parquet/{name}"
    )

print("All FAERS Silver tables saved successfully")

# CT File #

ct_bronze_final_path = "/Volumes/workspace/bronze/ct_parquet/cleaned/ct"
ct_bronze_df = spark.read.parquet(ct_bronze_final_path)

# BASE STANDARDIZATION #

ct_silver_df = ct_bronze_df.select(

    F.col("nct_id"),

    F.trim(
        F.col("title")
    ).alias("title"),

    F.when(
        F.col("status").isNull() | (F.trim(F.col("status")) == ""),
        "UNKNOWN"
    ).otherwise(
        F.regexp_replace(
            F.upper(
                F.trim(F.col("status"))
            ),
            "_",
            " "
        )
    ).alias("status"),

    F.when(
    F.col("phase").isNull(),
    "UNKNOWN"
).when(
    F.upper(F.trim(F.col("phase"))) == "NA",
    "NOT APPLICABLE"
).otherwise(
    F.regexp_replace(
        F.upper(F.trim(F.col("phase"))),
        "PHASE\\s*",
        "PHASE "
    )
).alias("phase"),

    F.col("start_date"),
    F.col("completion_date")
)

# SAVE SILVER LAYER TABLES #

ct_silver_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/silver/ct_parquet/ct"
)

print("All CT Silver tables saved successfully")